In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1) Train.csv 로드
df = pd.read_csv("Train.csv")

# 2) period_density 계산
def calc_period_density(text):
    s = str(text)
    if len(s) == 0:
        return 0.0
    return s.count(".") / len(s)

df["text"] = df["text"].fillna("")
df["period_density"] = df["text"].apply(calc_period_density)

print(f"전체 데이터 크기: {len(df)}")
print(f"period_density < 0.01 개수: {(df['period_density'] < 0.01).sum()}")

# ------------------------------
# [A] 원본 데이터에서의 정확도
# ------------------------------
X_full = df["text"]
y_full = df["label"]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

tfidf_full = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_f_vec = tfidf_full.fit_transform(X_train_f)
X_test_f_vec  = tfidf_full.transform(X_test_f)

lr_full = LogisticRegression(max_iter=1000)
lr_full.fit(X_train_f_vec, y_train_f)
y_pred_f = lr_full.predict(X_test_f_vec)
acc_full = accuracy_score(y_test_f, y_pred_f)

print(f"\nAccuracy on Full Data: {acc_full:.4f}")

# ------------------------------
# [B] 클린 + 중복 제거 데이터
# ------------------------------
df_clean = df[df["period_density"] >= 0.01].copy()
df_clean = df_clean.drop_duplicates(subset=["text"])

print(f"\nClean size (after period_density filter + dedup): {len(df_clean)}")

X_clean = df_clean["text"]
y_clean = df_clean["label"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

tfidf_clean = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_c_vec = tfidf_clean.fit_transform(X_train_c)
X_test_c_vec  = tfidf_clean.transform(X_test_c)

lr_clean = LogisticRegression(max_iter=1000)
lr_clean.fit(X_train_c_vec, y_train_c)
y_pred_c = lr_clean.predict(X_test_c_vec)
acc_clean = accuracy_score(y_test_c, y_pred_c)

print(f"Accuracy on Clean + Dedup Data: {acc_clean:.4f}")


전체 데이터 크기: 43398
period_density < 0.01 개수: 36203

Accuracy on Full Data: 0.8300

Clean size (after period_density filter + dedup): 6771
Accuracy on Clean + Dedup Data: 0.9668


In [6]:
import pandas as pd

# 1) Train.csv 로드
df = pd.read_csv("Train.csv")

# 2) period_density 계산 함수
def calc_period_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return text.count('.') / len(text)

df["period_density"] = df["text"].astype(str).apply(calc_period_density)

# 3) period_density >= 0.01 필터 + text 기준 중복 제거
df_clean = df[df["period_density"] >= 0.01].copy()
df_clean_dedup = df_clean.drop_duplicates(subset=["text"]).copy()

print("Clean + Dedup size:", len(df_clean_dedup))

# 4) label 분포 출력 (개수 + 비율)
print("\n[Label counts]")
print(df_clean_dedup["label"].value_counts())

print("\n[Label ratios]")
print(df_clean_dedup["label"].value_counts(normalize=True))


Clean + Dedup size: 6771

[Label counts]
label
1.0    4176
0.0    2595
Name: count, dtype: int64

[Label ratios]
label
1.0    0.616748
0.0    0.383252
Name: proportion, dtype: float64


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1) Train.csv 로드 (CWD가 data/raw 인 상태 가정)
df = pd.read_csv("Train.csv")

# 2) period_density 계산 함수 (이미 앞에서 썼다면 이 부분은 생략해도 됨)
def calc_period_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return text.count('.') / len(text)

df["period_density"] = df["text"].astype(str).apply(calc_period_density)

# 3) 노이즈 구간만 추출: period_density < 0.01
df_noisy = df[df["period_density"] < 0.01].copy()
print("Noisy size (raw):", len(df_noisy))

# 4) text 기준 중복 제거
df_noisy_dedup = df_noisy.drop_duplicates(subset=["text"]).copy()
print("Noisy size (after dedup):", len(df_noisy_dedup))

# 5) 라벨 분포 및 다수 클래스 baseline
label_counts = df_noisy_dedup["label"].value_counts()
label_ratios = df_noisy_dedup["label"].value_counts(normalize=True)

print("\n[Noisy Label counts]")
print(label_counts)

print("\n[Noisy Label ratios]")
print(label_ratios)

majority_ratio = label_ratios.max()
print(f"\n[Noisy Majority baseline accuracy] ≈ {majority_ratio:.4f}")

# 6) TF-IDF + Logistic Regression으로 성능 측정
X = df_noisy_dedup["text"].astype(str).values
y = df_noisy_dedup["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
acc_noisy = accuracy_score(y_test, y_pred)
print(f"\n[Accuracy on Noisy + Dedup Data] {acc_noisy:.4f}")


Noisy size (raw): 36203
Noisy size (after dedup): 34226

[Noisy Label counts]
label
1.0    17512
0.0    16714
Name: count, dtype: int64

[Noisy Label ratios]
label
1.0    0.511658
0.0    0.488342
Name: proportion, dtype: float64

[Noisy Majority baseline accuracy] ≈ 0.5117

[Accuracy on Noisy + Dedup Data] 0.7943


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1) 데이터 로드
df = pd.read_csv("Train.csv")  # CWD: /home/epistachio/projects/nlp_final_project/data/raw

# 2) period_density 계산
def calc_period_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return text.count('.') / len(text)

df["period_density"] = df["text"].astype(str).apply(calc_period_density)

# 3) text 기준 중복 제거 (전체 데이터에서 한 번만)
df_dedup = df.drop_duplicates(subset=["text"]).copy()
print("Total size after dedup:", len(df_dedup))

# 4) train/test split (전체 데이터 기준)
train_df, test_df = train_test_split(
    df_dedup,
    test_size=0.2,
    stratify=df_dedup["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

# 5) TF-IDF + Logistic Regression 학습
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(train_df["text"].astype(str))
X_test_vec = vectorizer.transform(test_df["text"].astype(str))

clf = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)
clf.fit(X_train_vec, train_df["label"])

# 6) 전체 test에 대한 성능
y_test = test_df["label"].values
y_pred = clf.predict(X_test_vec)

acc_all = accuracy_score(y_test, y_pred)
print(f"\n[Accuracy on ALL (dedup) test] {acc_all:.4f}")

# 7) test를 clean / noisy로 나누어 각각 성능 측정
mask_clean = test_df["period_density"] >= 0.01
mask_noisy = test_df["period_density"] < 0.01

print("\n[Test subset sizes]")
print("Clean test size:", mask_clean.sum())
print("Noisy test size:", mask_noisy.sum())

# clean test
if mask_clean.sum() > 0:
    acc_clean = accuracy_score(
        test_df.loc[mask_clean, "label"],
        y_pred[mask_clean.values]
    )
    print(f"[Accuracy on CLEAN test (period_density >= 0.01)] {acc_clean:.4f}")

# noisy test
if mask_noisy.sum() > 0:
    acc_noisy = accuracy_score(
        test_df.loc[mask_noisy, "label"],
        y_pred[mask_noisy.values]
    )
    print(f"[Accuracy on NOISY test (period_density < 0.01)] {acc_noisy:.4f}")


Total size after dedup: 40997
Train size: 32797
Test size: 8200

[Accuracy on ALL (dedup) test] 0.8227

[Test subset sizes]
Clean test size: 1364
Noisy test size: 6836
[Accuracy on CLEAN test (period_density >= 0.01)] 0.9765
[Accuracy on NOISY test (period_density < 0.01)] 0.7920


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1) 데이터 로드 및 period_density 계산
df = pd.read_csv("Train.csv")  # CWD: /home/epistachio/projects/nlp_final_project/data/raw

def calc_period_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return text.count('.') / len(text)

df["period_density"] = df["text"].astype(str).apply(calc_period_density)

# 2) text 기준 중복 제거
df_dedup = df.drop_duplicates(subset=["text"]).copy()

# 3) train/test split
train_df, test_df = train_test_split(
    df_dedup,
    test_size=0.2,
    stratify=df_dedup["label"],
    random_state=42
)

# 4) 모델 학습
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(train_df["text"].astype(str))
X_test_vec = vectorizer.transform(test_df["text"].astype(str))

clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train_vec, train_df["label"])

y_test = test_df["label"].values
y_pred = clf.predict(X_test_vec)

# 5) subset 나누기
mask_clean = test_df["period_density"] >= 0.01
mask_noisy = test_df["period_density"] < 0.01

def subset_stats(mask, name):
    y_true = y_test[mask.values]
    y_hat = y_pred[mask.values]
    size = mask.sum()
    if size == 0:
        return {"subset": name, "size": 0, "majority_baseline": None, "accuracy": None}
    
    # 다수 클래스 baseline
    majority_ratio = pd.Series(y_true).value_counts(normalize=True).max()
    acc = accuracy_score(y_true, y_hat)
    return {
        "subset": name,
        "size": int(size),
        "majority_baseline": float(majority_ratio),
        "accuracy": float(acc),
    }

rows = []
# 전체
rows.append(subset_stats(mask_clean | mask_noisy, "ALL (dedup test)"))
# 클린
rows.append(subset_stats(mask_clean, "CLEAN test (pd >= 0.01)"))
# 노이즈
rows.append(subset_stats(mask_noisy, "NOISY test (pd < 0.01)"))

summary_df = pd.DataFrame(rows)
print(summary_df)


                    subset  size  majority_baseline  accuracy
0         ALL (dedup test)  8200           0.529024  0.822683
1  CLEAN test (pd >= 0.01)  1364           0.605572  0.976540
2   NOISY test (pd < 0.01)  6836           0.513751  0.791984


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 1) Train.csv 로드
df = pd.read_csv("Train.csv")  # CWD: /home/epistachio/projects/nlp_final_project/data/raw

# 2) period_density 계산
def calc_period_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return text.count('.') / len(text)

df["period_density"] = df["text"].astype(str).apply(calc_period_density)

# 3) 노이즈 구간 추출 + 중복 제거
df_noisy = df[df["period_density"] < 0.01].copy()
df_noisy_dedup = df_noisy.drop_duplicates(subset=["text"]).copy()
print("Noisy size after dedup:", len(df_noisy_dedup))

# 4) 학습/테스트 분할
X = df_noisy_dedup["text"].astype(str).values
y = df_noisy_dedup["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 5) CountVectorizer (stop_words=None: 불용어 제거 안 함)
vectorizer = CountVectorizer(
    max_features=10000,
    stop_words=None
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 6) Multinomial Naive Bayes 학습
clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
acc = accuracy_score(y_test, y_pred)

# 7) 다수 클래스 baseline 비교
majority_ratio = pd.Series(y_test).value_counts(normalize=True).max()

print(f"[Naive Bayes accuracy on Noisy + Dedup] {acc:.4f}")
print(f"[Majority baseline on Noisy test] {majority_ratio:.4f}")


Noisy size after dedup: 34226
[Naive Bayes accuracy on Noisy + Dedup] 0.7832
[Majority baseline on Noisy test] 0.5117


In [11]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# 1) 데이터 로드 및 period_density 계산
df = pd.read_csv("Train.csv")  # CWD: /home/epistachio/projects/nlp_final_project/data/raw

def calc_period_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return text.count('.') / len(text)

df["period_density"] = df["text"].astype(str).apply(calc_period_density)

# 2) 노이즈 구간 + 중복 제거
df_noisy = df[df["period_density"] < 0.01].copy()
df_noisy_dedup = df_noisy.drop_duplicates(subset=["text"]).copy()
print("Noisy size after dedup:", len(df_noisy_dedup))

texts = df_noisy_dedup["text"].astype(str).values
labels = df_noisy_dedup["label"].values.astype(int)

# 3) CountVectorizer (불용어 제거 안 함)
vectorizer = CountVectorizer(stop_words=None, max_features=20000)
X = vectorizer.fit_transform(texts)
vocab = np.array(vectorizer.get_feature_names_out())

# 4) label별 단어 빈도 계산
mask0 = labels == 0
mask1 = labels == 1

word_counts0 = np.array(X[mask0].sum(axis=0)).ravel()
word_counts1 = np.array(X[mask1].sum(axis=0)).ravel()

top_k = 100

idx0 = word_counts0.argsort()[::-1][:top_k]
idx1 = word_counts1.argsort()[::-1][:top_k]

top_words0 = list(zip(vocab[idx0], word_counts0[idx0]))
top_words1 = list(zip(vocab[idx1], word_counts1[idx1]))

print("\n[Top 20 words in noisy label 0]")
print(top_words0[:20])

print("\n[Top 20 words in noisy label 1]")
print(top_words1[:20])

# 5) 상위 100개 교집합 크기
set0 = set(vocab[idx0])
set1 = set(vocab[idx1])
common = set0 & set1

print(f"\nTop {top_k} 교집합 단어 개수:", len(common))
print("일부 공통 단어 예시:", sorted(list(common))[:20])


Noisy size after dedup: 34226

[Top 20 words in noisy label 0]
[('the', np.int64(243494)), ('to', np.int64(132688)), ('of', np.int64(106782)), ('and', np.int64(103296)), ('in', np.int64(77105)), ('that', np.int64(71761)), ('is', np.int64(50111)), ('for', np.int64(43278)), ('it', np.int64(39188)), ('on', np.int64(38595)), ('trump', np.int64(37032)), ('he', np.int64(36552)), ('with', np.int64(30245)), ('was', np.int64(29335)), ('his', np.int64(28074)), ('as', np.int64(27956)), ('this', np.int64(27348)), ('be', np.int64(24436)), ('by', np.int64(22960)), ('they', np.int64(22887))]

[Top 20 words in noisy label 1]
[('the', np.int64(255439)), ('to', np.int64(130635)), ('of', np.int64(110595)), ('in', np.int64(97647)), ('and', np.int64(97393)), ('on', np.int64(57405)), ('said', np.int64(50792)), ('that', np.int64(47364)), ('for', np.int64(44242)), ('with', np.int64(29808)), ('he', np.int64(29175)), ('is', np.int64(28778)), ('by', np.int64(27271)), ('as', np.int64(26551)), ('it', np.int64(2571

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1) 데이터 로드
df = pd.read_csv("Train.csv")  # CWD: /home/epistachio/projects/nlp_final_project/data/raw

# 2) period_density 없으면 다시 계산
def calc_period_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return text.count('.') / len(text)

if "period_density" not in df.columns:
    df["period_density"] = df["text"].astype(str).apply(calc_period_density)

# 3) period_density 하위 32% 임계값 찾기
q32 = df["period_density"].quantile(0.32)
print("32% quantile of period_density:", q32)

# 4) 이 기준으로 '진짜 노이즈(32%)' 후보 추출 + 중복 제거
df_noise32 = df[df["period_density"] <= q32].copy()
df_noise32_dedup = df_noise32.drop_duplicates(subset=["text"]).copy()

print("noise32 size (after dedup):", len(df_noise32_dedup))

# 5) 라벨 분포 및 다수 클래스 baseline
label_counts = df_noise32_dedup["label"].value_counts()
label_ratios = df_noise32_dedup["label"].value_counts(normalize=True)

print("\n[noise32 Label counts]")
print(label_counts)

print("\n[noise32 Label ratios]")
print(label_ratios)

majority_ratio = label_ratios.max()
print(f"\n[noise32 Majority baseline accuracy] ≈ {majority_ratio:.4f}")

# 6) 이 32% 구간에 대해서만 LR 성능 측정
X = df_noise32_dedup["text"].astype(str).values
y = df_noise32_dedup["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
acc_noise32 = accuracy_score(y_test, y_pred)
print(f"\n[Accuracy on noise32 (hard 32% by period_density)] {acc_noise32:.4f}")


32% quantile of period_density: 0.0
noise32 size (after dedup): 13849

[noise32 Label counts]
label
1.0    6954
0.0    6895
Name: count, dtype: int64

[noise32 Label ratios]
label
1.0    0.50213
0.0    0.49787
Name: proportion, dtype: float64

[noise32 Majority baseline accuracy] ≈ 0.5021

[Accuracy on noise32 (hard 32% by period_density)] 0.5022


In [13]:
import pandas as pd
import re

# 1) 데이터 로드 + period_density (이미 있으면 생략 가능)
df = pd.read_csv("Train.csv")

def calc_period_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return text.count('.') / len(text)

if "period_density" not in df.columns:
    df["period_density"] = df["text"].astype(str).apply(calc_period_density)

# 2) 규칙 정의 -------------------------

# (1) 문장 끝 글자 규칙: !, :, ?
def rule_end_punct(text):
    text = text.rstrip()
    return text.endswith('!') or text.endswith(':') or text.endswith('?')

# (2) garbage token: 문장 중간에 "외톨이 소문자" 토큰 (예: ' b ', ' z ')
#   - 네가 실제로 봤던 패턴에 맞게 문자 집합은 필요하면 좁혀도 됨.
garbage_pattern = re.compile(r'\b[a-z]\b')

def rule_garbage_token(text):
    return bool(garbage_pattern.search(text))

# 3) 규칙별 마스크 ---------------------

texts = df["text"].astype(str)

mask_end = texts.apply(rule_end_punct)
mask_garbage = texts.apply(rule_garbage_token)
mask_any_rule = mask_end | mask_garbage

print("Total samples:", len(df))
print("rule_end_punct count:", mask_end.sum())
print("rule_garbage_token count:", mask_garbage.sum())
print("any_rule count (union):", mask_any_rule.sum())

# 4) 규칙이 잡은 구간에서의 라벨 분포 ----------------
def show_rule_stats(mask, name):
    sub = df[mask]
    if len(sub) == 0:
        print(f"\n[{name}] no samples")
        return
    counts = sub["label"].value_counts()
    ratios = sub["label"].value_counts(normalize=True)
    print(f"\n[{name}] size:", len(sub))
    print("[label counts]\n", counts)
    print("[label ratios]\n", ratios)

    # noise32(마침표 0) 포함 비율
    noise32 = sub[sub["period_density"] == 0.0]
    print(f"  - among them, period_density==0 size: {len(noise32)} "
          f"({len(noise32) / len(sub):.3f} of this rule group)")

show_rule_stats(mask_end, "rule_end_punct")
show_rule_stats(mask_garbage, "rule_garbage_token")
show_rule_stats(mask_any_rule, "ANY rule")


Total samples: 43398
rule_end_punct count: 1486
rule_garbage_token count: 32235
any_rule count (union): 32408

[rule_end_punct] size: 1486
[label counts]
 label
0.0    1466
1.0      20
Name: count, dtype: int64
[label ratios]
 label
0.0    0.986541
1.0    0.013459
Name: proportion, dtype: float64
  - among them, period_density==0 size: 242 (0.163 of this rule group)

[rule_garbage_token] size: 32235
[label counts]
 label
0.0    16140
1.0    16095
Name: count, dtype: int64
[label ratios]
 label
0.0    0.500698
1.0    0.499302
Name: proportion, dtype: float64
  - among them, period_density==0 size: 3155 (0.098 of this rule group)

[ANY rule] size: 32408
[label counts]
 label
0.0    16311
1.0    16097
Name: count, dtype: int64
[label ratios]
 label
0.0    0.503302
1.0    0.496698
Name: proportion, dtype: float64
  - among them, period_density==0 size: 3273 (0.101 of this rule group)
